# Huấn luyện YOLO11m Hybrid (VinDr + NIH) cho CheXNet

Notebook này huấn luyện **YOLO11m** để phát hiện tổn thương trên X-quang ngực,
sử dụng bộ dữ liệu **Hybrid** kết hợp:
- **VinDr-CXR**: BBox chuẩn từ bác sĩ (10 bệnh có bbox)
- **NIH ChestX-ray14**: BBox sinh tự động (Pseudo-labels) từ **CheXNet Hybrid Model** cho 4 bệnh VinDr không có

### Các cải tiến so với bản trước:
1. **Knowledge Distillation** — CheXNet Attention Map sinh Pseudo BBox cho Pneumonia, Edema, Emphysema, Hernia
2. **Dữ liệu Hybrid** — VinDr (~18K) + NIH (~112K ảnh), YOLO học từ cả 2 bộ
3. **WBF trên Ground Truth** — Gộp boxes từ nhiều bác sĩ (VinDr) thành consensus box
4. **Progressive Training** — Phase 1 (640px) rồi Phase 2 (1024px)
5. **TTA + WBF Inference** — Dự đoán ảnh gốc + flip, gộp WBF
6. **Cascade 2 giai đoạn** — `Final_Score = P_classifier * Conf_YOLO`
7. **14 classes đồng bộ CheXNet** — Mapping trực tiếp với CheXNet classifier

### Chuẩn bị trên Kaggle:
1. Bật **GPU T4 x2** (Settings > Accelerator)
2. **Add Data**: Dataset `cuonglevc1/xraydata` (chứa data_1..10, CSV files)
3. **Add Data**: Upload `hybrid_model_light.pth` + `Model.py` + `checkpoint_utils.py`

In [ ]:
!pip install -q ultralytics ensemble-boxes scikit-learn timm

import os
import sys
import cv2
import glob
import shutil
import gc
import numpy as np
import pandas as pd
from collections import Counter, defaultdict
from tqdm.auto import tqdm
from ensemble_boxes import weighted_boxes_fusion
from ultralytics import YOLO
import torch
import torch.nn.functional as F
import torchvision.transforms as transforms
from PIL import Image

# ======================================================
# CAU HINH DUONG DAN (SUA CHO KHOP VOI KAGGLE CUA BAN)
# ======================================================

# Dataset anh VinDr + NIH (da gop, chia 10 thu muc PNG)
DATA_ROOT = '/kaggle/input/datasets/cuonglevc1/xraydata'

# CSV files
TRAIN_CSV = os.path.join(DATA_ROOT, 'train_list.csv')
VAL_CSV   = os.path.join(DATA_ROOT, 'val_list.csv')
TEST_CSV  = os.path.join(DATA_ROOT, 'test_list.csv')

# CheXNet Model code (Model.py, checkpoint_utils.py)
CHEXNET_CODE_DIR   = '/kaggle/input/datasets/cuonglevc1/modelv'
# CheXNet Model weights
CHEXNET_MODEL_PATH = '/kaggle/input/models/cuonglevc1/chex/pytorch/default/1/hybrid_model_light.pth'

# Thu muc lam viec (output)
WORK_DIR = '/kaggle/working/yolo_hybrid'
os.makedirs(WORK_DIR, exist_ok=True)

# Detect GPU
n_gpus = torch.cuda.device_count()
for i in range(n_gpus):
    props = torch.cuda.get_device_properties(i)
    mem = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
    print(f'GPU {i}: {torch.cuda.get_device_name(i)} ({mem / 1e9:.1f} GB)')
DEVICE = list(range(n_gpus)) if n_gpus > 1 else 0
GPU_DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'YOLO device: {DEVICE}  |  CheXNet device: {GPU_DEVICE}')

## 1. Ánh xạ 14 Class YOLO - CheXNet

YOLO sẽ có **14 classes** (bỏ `No Finding`), đồng bộ **1-1** với CheXNet classifier:

| YOLO ID | Tên bệnh | Nguồn BBox | CheXNet idx |
|---------|-----------|------------|-------------|
| 0 | Atelectasis | VinDr GT | 1 |
| 1 | Cardiomegaly | VinDr GT | 2 |
| 2 | Effusion | VinDr GT | 3 |
| 3 | Infiltration | VinDr GT | 4 |
| 4 | Mass | VinDr GT | 5 |
| 5 | Nodule | VinDr GT | 6 |
| 6 | **Pneumonia** | **CheXNet Pseudo** | 7 |
| 7 | Pneumothorax | VinDr GT | 8 |
| 8 | Consolidation | VinDr GT | 9 |
| 9 | **Edema** | **CheXNet Pseudo** | 10 |
| 10 | **Emphysema** | **CheXNet Pseudo** | 11 |
| 11 | Fibrosis | VinDr GT | 12 |
| 12 | Pleural_Thickening | VinDr GT | 13 |
| 13 | **Hernia** | **CheXNet Pseudo** | 14 |

> 4 bệnh in đậm: VinDr không có BBox, dùng CheXNet Attention Map sinh Pseudo-labels từ ảnh NIH.

In [ ]:
# -- 14 YOLO Classes (dong bo CheXNet, bo No Finding) --
YOLO_CLASSES = [
    'Atelectasis',        # 0
    'Cardiomegaly',       # 1
    'Effusion',           # 2
    'Infiltration',       # 3
    'Mass',               # 4
    'Nodule',             # 5
    'Pneumonia',          # 6  <- Pseudo from NIH
    'Pneumothorax',       # 7
    'Consolidation',      # 8
    'Edema',              # 9  <- Pseudo from NIH
    'Emphysema',          # 10 <- Pseudo from NIH
    'Fibrosis',           # 11
    'Pleural_Thickening', # 12
    'Hernia',             # 13 <- Pseudo from NIH
]

# Mapping ten benh trong CSV -> YOLO class ID
DISEASE_TO_YOLO_ID = {name: i for i, name in enumerate(YOLO_CLASSES)}

# 4 benh can sinh Pseudo-label (VinDr khong co BBox)
PSEUDO_DISEASES = ['Pneumonia', 'Edema', 'Emphysema', 'Hernia']
PSEUDO_YOLO_IDS = {d: DISEASE_TO_YOLO_ID[d] for d in PSEUDO_DISEASES}

# CheXNet class index cho moi pseudo disease (de lay attention map channel)
CHEXNET_CLASS_NAMES = [
    'No Finding', 'Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration',
    'Mass', 'Nodule', 'Pneumonia', 'Pneumothorax', 'Consolidation',
    'Edema', 'Emphysema', 'Fibrosis', 'Pleural_Thickening', 'Hernia'
]
PSEUDO_CHEXNET_IDX = {d: CHEXNET_CLASS_NAMES.index(d) for d in PSEUDO_DISEASES}
print(f'Pseudo disease -> CheXNet channel: {PSEUDO_CHEXNET_IDX}')

# -- Doc CSV --
df_train = pd.read_csv(TRAIN_CSV)
df_val   = pd.read_csv(VAL_CSV)
print(f'\nTrain: {len(df_train)} rows, unique images: {df_train["Image Index"].nunique()}')
print(f'Val:   {len(df_val)} rows, unique images: {df_val["Image Index"].nunique()}')

for split_name, df in [('Train', df_train), ('Val', df_val)]:
    print(f'\n{split_name} source distribution:')
    print(df.groupby('Source')['Image Index'].nunique())

# -- Xay dung Image Path Map O(1) --
print('\nDang xay dung Image Path Map...')
image_path_map = {}

# Debug: In cau truc thu muc
print(f'DATA_ROOT = {DATA_ROOT}')
print(f'Contents of DATA_ROOT:')
for item in sorted(os.listdir(DATA_ROOT)):
    full = os.path.join(DATA_ROOT, item)
    if os.path.isdir(full):
        sub_items = os.listdir(full)
        print(f'  [DIR]  {item}/ ({len(sub_items)} items, first 3: {sub_items[:3]})')
    else:
        print(f'  [FILE] {item} ({os.path.getsize(full)} bytes)')

# Scan TAT CA thu muc con (de quy) trong DATA_ROOT
# Dung os.walk de tim tat ca file anh o bat ky cap nao
IMG_EXTENSIONS = {'.png', '.jpg', '.jpeg'}
for root, dirs, files in os.walk(DATA_ROOT):
    for f in files:
        ext = os.path.splitext(f)[1].lower()
        if ext in IMG_EXTENSIONS:
            image_path_map[f] = os.path.join(root, f)

print(f'Indexed {len(image_path_map):,} images')

# Neu van 0, thu quet khong loc extension
if len(image_path_map) == 0:
    print('WARNING: 0 images found with .png/.jpg/.jpeg extension!')
    print('Thu quet tat ca files (khong loc extension)...')
    all_files_sample = []
    for root, dirs, files in os.walk(DATA_ROOT):
        for f in files[:5]:
            all_files_sample.append(os.path.join(root, f))
        if len(all_files_sample) >= 20:
            break
    print(f'Sample files found: {all_files_sample}')
    
    # Fallback: quet tat ca files, bo qua CSV
    for root, dirs, files in os.walk(DATA_ROOT):
        for f in files:
            if not f.endswith('.csv'):
                image_path_map[f] = os.path.join(root, f)
    print(f'After fallback scan: {len(image_path_map):,} files indexed')

for split_name, df in [('Train', df_train), ('Val', df_val)]:
    filenames = df['Image Index'].unique()
    found = sum(1 for f in filenames if f in image_path_map)
    missing = len(filenames) - found
    print(f'{split_name}: {found}/{len(filenames)} found, {missing} missing')
    if missing > 0 and found == 0:
        print(f'  Sample CSV filenames: {list(filenames[:5])}')
        print(f'  Sample indexed keys: {list(image_path_map.keys())[:5]}')

## 2. Hợp nhất Ground Truth bằng WBF (VinDr)

Trong VinDr-CXR, **mỗi ảnh được nhiều bác sĩ** (cột `rad_id`) đánh dấu độc lập.
Dùng **Weighted Boxes Fusion (WBF)** gộp các boxes trùng lặp thành 1 consensus box.

In [ ]:
def fuse_vindr_boxes(group_df, img_w, img_h):
    """Gop boxes tu nhieu bac si cho 1 anh VinDr bang WBF."""
    has_bbox = group_df['x_min'].notna() & group_df['x_max'].notna()
    bbox_df = group_df[has_bbox].copy()
    
    if len(bbox_df) == 0:
        return []
    
    rads = bbox_df['rad_id'].dropna().unique()
    
    if len(rads) <= 1:
        results = []
        for _, row in bbox_df.iterrows():
            disease = row['Finding Labels']
            if disease not in DISEASE_TO_YOLO_ID:
                continue
            results.append({
                'yolo_class_id': DISEASE_TO_YOLO_ID[disease],
                'x_min': float(row['x_min']), 'y_min': float(row['y_min']),
                'x_max': float(row['x_max']), 'y_max': float(row['y_max']),
            })
        return results
    
    boxes_list, scores_list, labels_list = [], [], []
    for rad_id in rads:
        rad_df = bbox_df[bbox_df['rad_id'] == rad_id]
        boxes, scores, labels = [], [], []
        for _, row in rad_df.iterrows():
            disease = row['Finding Labels']
            if disease not in DISEASE_TO_YOLO_ID:
                continue
            x1 = np.clip(float(row['x_min']) / img_w, 0, 1)
            y1 = np.clip(float(row['y_min']) / img_h, 0, 1)
            x2 = np.clip(float(row['x_max']) / img_w, 0, 1)
            y2 = np.clip(float(row['y_max']) / img_h, 0, 1)
            if x2 <= x1 or y2 <= y1:
                continue
            boxes.append([x1, y1, x2, y2])
            scores.append(1.0)
            labels.append(DISEASE_TO_YOLO_ID[disease])
        if boxes:
            boxes_list.append(boxes)
            scores_list.append(scores)
            labels_list.append(labels)
    
    if not boxes_list:
        return []
    
    fused_boxes, fused_scores, fused_labels = weighted_boxes_fusion(
        boxes_list, scores_list, labels_list,
        weights=[1.0] * len(boxes_list), iou_thr=0.5, skip_box_thr=0.0001
    )
    results = []
    for box, label in zip(fused_boxes, fused_labels):
        results.append({
            'yolo_class_id': int(label),
            'x_min': box[0] * img_w, 'y_min': box[1] * img_h,
            'x_max': box[2] * img_w, 'y_max': box[3] * img_h,
        })
    return results

print('WBF function ready.')

## 3. Knowledge Distillation: CheXNet sinh Pseudo-Labels bằng Grad-CAM

### Vấn đề
VinDr-CXR **không có BBox** cho 4 bệnh: Pneumonia, Edema, Emphysema, Hernia.
Attention maps (output từ attention head) **gần = 0** trên ảnh NIH vì chúng chỉ
được train bằng Dice Loss trên ảnh VinDr có bbox ground truth.

### Giải pháp: Grad-CAM
Dùng **Grad-CAM** trên `fpn_features` (layer cuối trước pooling):
1. Forward pass qua CheXNet → lấy feature maps từ FPN merge layer
2. Backprop từ **logit của bệnh target** → lấy gradients
3. **Grad-CAM = ReLU(sum(weights × features))** — weights = mean(gradients)
4. Threshold (Otsu / percentile) → findContours → sinh BBox
5. Lọc bbox theo diện tích (> 0.1% ảnh)

> **Tại sao Grad-CAM hoạt động?** Vì classification head **ĐÃ được train trên NIH**
> (BCE loss trên tất cả 15 bệnh). Gradients phản ánh vùng nào trên ảnh đóng góp
> nhiều nhất vào dự đoán bệnh → đó chính là vùng tổn thương.

In [ ]:
# ======================================
# BUOC 3.1: Load CheXNet Hybrid Model
# ======================================
if CHEXNET_CODE_DIR not in sys.path:
    sys.path.insert(0, CHEXNET_CODE_DIR)

from Model import HybridCNNViTModel
from checkpoint_utils import load_checkpoint_safe, extract_state_dict

print('Loading CheXNet Hybrid Model...')
ckpt = load_checkpoint_safe(CHEXNET_MODEL_PATH, device=torch.device('cpu'))
model_size = ckpt.get('model_size', 'base')
img_size = ckpt.get('img_size', 384)
cleaned_sd = extract_state_dict(ckpt)

chexnet_model = HybridCNNViTModel(num_classes=15, model_size=model_size,
                                   img_size=img_size, pretrained=False)
chexnet_model.load_state_dict(cleaned_sd, strict=False)
chexnet_model = chexnet_model.to(GPU_DEVICE).eval()
if hasattr(chexnet_model, 'set_grad_checkpointing'):
    chexnet_model.set_grad_checkpointing(False)

chexnet_transform = transforms.Compose([
    transforms.Resize(int(img_size * 1.14)),
    transforms.CenterCrop(img_size),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

del ckpt, cleaned_sd
gc.collect()
torch.cuda.empty_cache()
print(f'CheXNet loaded! Size={model_size}, ImgSize={img_size}')

# ======================================
# Grad-CAM: Sinh heatmap tu classification gradients
# ======================================
# Attention maps (attention_head output) gan = 0 tren anh NIH
# vi chung chi duoc train (Dice Loss) tren anh VinDr co bbox.
# => Dung Grad-CAM tren fpn_features (duoc dung boi classifier)
# Classification head DA duoc train tren NIH => Grad-CAM se co y nghia.

# Hook vao fpn_merge (layer cuoi truoc attention head va pooling)
_gradcam_features = {}
_gradcam_grads = {}

def _save_features(module, input, output):
    _gradcam_features['value'] = output

def _save_grads(module, grad_input, grad_output):
    _gradcam_grads['value'] = grad_output[0]

hook_fwd = chexnet_model.fpn_merge.register_forward_hook(_save_features)
hook_bwd = chexnet_model.fpn_merge.register_full_backward_hook(_save_grads)
print('Grad-CAM hooks registered on fpn_merge layer.')

In [ ]:
# ======================================
# BUOC 3.2: Sinh Pseudo-Labels bang Grad-CAM
# ======================================

def gradcam_to_bboxes(gradcam_map, img_w, img_h, min_area_ratio=0.001):
    """Chuyen Grad-CAM heatmap thanh danh sach YOLO bboxes."""
    if gradcam_map.max() < 1e-6:
        return []
    
    map_resized = cv2.resize(gradcam_map, (img_w, img_h), interpolation=cv2.INTER_CUBIC)
    map_norm = (map_resized - map_resized.min()) / (map_resized.max() - map_resized.min() + 1e-8)
    map_u8 = np.clip(map_norm * 255, 0, 255).astype(np.uint8)
    
    min_area = min_area_ratio * img_w * img_h
    
    # Thu nhieu phuong phap threshold
    for thresh_val in [None, np.percentile(map_u8[map_u8 > 0], 50) if (map_u8 > 0).any() else 128,
                       map_u8.max() * 0.3, map_u8.max() * 0.2]:
        if thresh_val is None:
            _, binary = cv2.threshold(map_u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        else:
            _, binary = cv2.threshold(map_u8, int(thresh_val), 255, cv2.THRESH_BINARY)
        
        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        bboxes = []
        for cnt in contours:
            if cv2.contourArea(cnt) < min_area:
                continue
            x, y, w, h = cv2.boundingRect(cnt)
            roi = map_norm[y:y+h, x:x+w]
            mean_act = float(roi.mean()) if roi.size > 0 else 0
            bboxes.append({'x_min': x, 'y_min': y, 'x_max': x+w, 'y_max': y+h,
                           'confidence': mean_act})
        if bboxes:
            bboxes.sort(key=lambda b: b['confidence'], reverse=True)
            return bboxes[:3]
    return []


def compute_gradcam_single(image_tensor, target_class_idx):
    """
    Tinh Grad-CAM cho 1 anh duy nhat (tiet kiem VRAM toi da).
    
    Args:
        image_tensor: [3, H, W] tensor (chua unsqueeze)
        target_class_idx: int, CheXNet class index
    Returns:
        numpy [H_feat, W_feat] gradcam map (normalized 0-1), or None if failed
    """
    try:
        chexnet_model.zero_grad()
        # Clear hooks
        _gradcam_features.clear()
        _gradcam_grads.clear()
        
        inp = image_tensor.unsqueeze(0).to(GPU_DEVICE)  # [1, 3, H, W]
        inp.requires_grad_(True)
        
        # Forward (float32 — mixed precision cho backward ko stable)
        logits, _ = chexnet_model(inp)
        
        # Backward chi tu 1 logit
        score = logits[0, target_class_idx]
        score.backward()
        
        # Lay features va grads tu hooks
        features = _gradcam_features['value'].detach()  # [1, C, H, W]
        grads = _gradcam_grads['value'].detach()         # [1, C, H, W]
        
        # Grad-CAM
        weights = grads.mean(dim=[2, 3], keepdim=True)   # [1, C, 1, 1]
        cam = (weights * features).sum(dim=1)            # [1, H, W]
        cam = torch.relu(cam)
        
        c = cam[0].cpu().numpy()
        if c.max() > 0:
            c = c / c.max()
        
        # Giai phong VRAM ngay lap tuc
        del inp, logits, score, features, grads, weights, cam
        
        return c
    except Exception as e:
        print(f'    Grad-CAM error: {e}')
        return None
    finally:
        # Luon giai phong bo nho
        chexnet_model.zero_grad()
        _gradcam_features.clear()
        _gradcam_grads.clear()
        torch.cuda.empty_cache()


# Debug stats
_debug_stats = {'total': 0, 'has_bbox': 0, 'cam_maxes': [], 'cam_means': [],
                'errors': 0}

def generate_pseudo_labels_gradcam(df, split_name):
    """Sinh pseudo-labels bang Grad-CAM (xu ly tung anh 1 de tranh OOM)."""
    global _debug_stats
    _debug_stats = {'total': 0, 'has_bbox': 0, 'cam_maxes': [], 'cam_means': [],
                    'errors': 0}
    
    nih_df = df[df['Source'] == 'NIH'].copy()
    pseudo_mask = nih_df[PSEUDO_DISEASES].max(axis=1) > 0
    target_df = nih_df[pseudo_mask].drop_duplicates(subset='Image Index')
    
    print(f'\n[{split_name}] Anh NIH can pseudo-label: {len(target_df)}')
    if len(target_df) == 0:
        return {}
    
    pseudo_labels = {}
    
    for disease in PSEUDO_DISEASES:
        chex_idx = PSEUDO_CHEXNET_IDX[disease]
        yolo_id = DISEASE_TO_YOLO_ID[disease]
        
        disease_df = target_df[target_df[disease] == 1]
        if len(disease_df) == 0:
            continue
        
        print(f'  [{disease}] {len(disease_df)} images, CheXNet idx={chex_idx}, YOLO id={yolo_id}')
        err_count = 0
        
        for _, row in tqdm(disease_df.iterrows(), total=len(disease_df),
                           desc=f'  [{disease}]', leave=False):
            filename = row['Image Index']
            img_path = image_path_map.get(filename)
            if img_path is None:
                continue
            
            try:
                pil_img = Image.open(img_path).convert('RGB')
                img_w, img_h = pil_img.size
                tensor = chexnet_transform(pil_img)
                del pil_img
            except Exception:
                continue
            
            cam = compute_gradcam_single(tensor, chex_idx)
            del tensor
            
            if cam is None:
                _debug_stats['errors'] += 1
                err_count += 1
                if err_count >= 3:
                    print(f'    WARNING: {err_count} consecutive errors, skipping rest of {disease}')
                    break
                continue
            err_count = 0  # Reset on success
            
            _debug_stats['total'] += 1
            _debug_stats['cam_maxes'].append(float(cam.max()))
            _debug_stats['cam_means'].append(float(cam.mean()))
            
            bboxes = gradcam_to_bboxes(cam, img_w, img_h)
            if bboxes:
                _debug_stats['has_bbox'] += 1
                new_boxes = [(yolo_id, b['x_min'], b['y_min'],
                              b['x_max'], b['y_max']) for b in bboxes]
                if filename in pseudo_labels:
                    pseudo_labels[filename].extend(new_boxes)
                else:
                    pseudo_labels[filename] = new_boxes
            del cam
        
        # Giup GC giua cac benh
        gc.collect()
        torch.cuda.empty_cache()
    
    print(f'\n[{split_name}] === GRAD-CAM STATISTICS ===')
    print(f'  Total Grad-CAM maps: {_debug_stats["total"]}')
    print(f'  Maps that produced bbox: {_debug_stats["has_bbox"]}')
    print(f'  Errors: {_debug_stats["errors"]}')
    if _debug_stats['cam_maxes']:
        maxes = np.array(_debug_stats['cam_maxes'])
        means = np.array(_debug_stats['cam_means'])
        print(f'  CAM max  - min:{maxes.min():.4f} mean:{maxes.mean():.4f} max:{maxes.max():.4f}')
        print(f'  CAM mean - min:{means.min():.6f} mean:{means.mean():.6f} max:{means.max():.6f}')
        print(f'  % maps with max > 0.1: {(maxes > 0.1).mean():.1%}')
        print(f'  % maps with max > 0.5: {(maxes > 0.5).mean():.1%}')
    
    print(f'[{split_name}] Pseudo-labels sinh duoc cho {len(pseudo_labels)} anh')
    return pseudo_labels

# -- Chay sinh pseudo-labels --
pseudo_train = generate_pseudo_labels_gradcam(df_train, 'Train')
pseudo_val   = generate_pseudo_labels_gradcam(df_val, 'Val')

# -- Go hooks va giai phong GPU --
hook_fwd.remove()
hook_bwd.remove()
del chexnet_model
gc.collect()
torch.cuda.empty_cache()
print('\nCheXNet model unloaded. GPU freed for YOLO training.')

## 4. Chuẩn bị YOLO Dataset (Symlink + Labels)

**Chiến lược tiết kiệm disk** (Kaggle chi có ~20GB /kaggle/working):
- **Ảnh**: Tạo **symlink** thay vì copy (tốn ~0 bytes!)
- **Labels**: Ghi file `.txt` (vài KB mỗi file)
- YOLO Ultralytics tự match `images/train/xxx.png` <-> `labels/train/xxx.txt`

**Nguồn labels:**
- VinDr: BBox từ CSV (qua WBF fuse)
- NIH (pseudo): BBox từ CheXNet attention
- No Finding: File `.txt` RỖNG (negative examples)

In [ ]:
# ======================================
# BUOC 4: Tao YOLO Dataset
# ======================================

images_dir = os.path.join(WORK_DIR, 'images')
labels_dir = os.path.join(WORK_DIR, 'labels')
for split in ['train', 'val']:
    os.makedirs(os.path.join(images_dir, split), exist_ok=True)
    os.makedirs(os.path.join(labels_dir, split), exist_ok=True)

def process_split(df, split, pseudo_labels):
    """Xu ly 1 split: symlink anh + ghi YOLO labels."""
    stats = Counter()
    grouped = df.groupby('Image Index', sort=False)
    
    for img_name, group in tqdm(grouped, desc=f'Processing {split}', total=len(grouped)):
        img_path = image_path_map.get(img_name)
        if img_path is None:
            stats['missing'] += 1
            continue
        
        source = group['Source'].iloc[0]
        
        # -- Tao symlink anh --
        link_path = os.path.join(images_dir, split, img_name)
        if not os.path.exists(link_path):
            try:
                os.symlink(img_path, link_path)
            except OSError:
                try:
                    os.link(img_path, link_path)
                except OSError:
                    shutil.copy2(img_path, link_path)
        
        # -- Tao label file --
        label_path = os.path.join(labels_dir, split, img_name.rsplit('.', 1)[0] + '.txt')
        lines = []
        
        if source == 'VinDr-CXR':
            has_any_bbox = group['x_min'].notna().any()
            if has_any_bbox:
                try:
                    with Image.open(img_path) as pil:
                        img_w, img_h = pil.size
                except Exception:
                    stats['read_error'] += 1
                    continue
                
                fused = fuse_vindr_boxes(group, img_w, img_h)
                for box in fused:
                    cls_id = box['yolo_class_id']
                    x_c = np.clip(((box['x_min'] + box['x_max']) / 2) / img_w, 0, 1)
                    y_c = np.clip(((box['y_min'] + box['y_max']) / 2) / img_h, 0, 1)
                    bw = np.clip((box['x_max'] - box['x_min']) / img_w, 0, 1)
                    bh = np.clip((box['y_max'] - box['y_min']) / img_h, 0, 1)
                    lines.append(f'{cls_id} {x_c:.6f} {y_c:.6f} {bw:.6f} {bh:.6f}')
                stats['vindr_bbox'] += 1
            else:
                stats['vindr_nofinding'] += 1
        
        elif source == 'NIH':
            if img_name in pseudo_labels:
                try:
                    with Image.open(img_path) as pil:
                        img_w, img_h = pil.size
                except Exception:
                    stats['read_error'] += 1
                    continue
                
                for (cls_id, x1, y1, x2, y2) in pseudo_labels[img_name]:
                    x_c = np.clip(((x1 + x2) / 2) / img_w, 0, 1)
                    y_c = np.clip(((y1 + y2) / 2) / img_h, 0, 1)
                    bw = np.clip((x2 - x1) / img_w, 0, 1)
                    bh = np.clip((y2 - y1) / img_h, 0, 1)
                    lines.append(f'{cls_id} {x_c:.6f} {y_c:.6f} {bw:.6f} {bh:.6f}')
                stats['nih_pseudo'] += 1
            else:
                stats['nih_nofinding'] += 1
        
        with open(label_path, 'w') as f:
            f.write('\n'.join(lines))
    
    return stats

print('=== Processing Train Split ===')
train_stats = process_split(df_train, 'train', pseudo_train)
print(f'Train stats: {dict(train_stats)}')

print('\n=== Processing Val Split ===')
val_stats = process_split(df_val, 'val', pseudo_val)
print(f'Val stats: {dict(val_stats)}')

for split in ['train', 'val']:
    label_files = glob.glob(f'{labels_dir}/{split}/*.txt')
    non_empty = sum(1 for f in label_files if os.path.getsize(f) > 0)
    print(f'\n{split}: {len(label_files)} label files ({non_empty} co bbox, '
          f'{len(label_files) - non_empty} negative)')

## 5. Tạo file `data.yaml`

14 classes đồng bộ CheXNet (bỏ No Finding).

In [ ]:
yaml_content = f"""path: {WORK_DIR}
train: images/train
val: images/val

nc: 14

names:
  0: Atelectasis
  1: Cardiomegaly
  2: Effusion
  3: Infiltration
  4: Mass
  5: Nodule
  6: Pneumonia
  7: Pneumothorax
  8: Consolidation
  9: Edema
  10: Emphysema
  11: Fibrosis
  12: Pleural_Thickening
  13: Hernia
"""

yaml_path = os.path.join(WORK_DIR, 'data.yaml')
with open(yaml_path, 'w') as f:
    f.write(yaml_content)
print(f'data.yaml saved: {yaml_path}')
print(yaml_content)

## 6. Huấn luyện Progressive

**Phase 1** (640px, 25 epochs): Học đặc trưng cơ bản
**Phase 2** (1024px, 55 epochs): Fine-tune ở độ phân giải cao

Augmentation cho X-quang: Tắt flipud, mosaic, mixup. Giữ fliplr, degrees, scale.

In [ ]:
# ======================================
# PHASE 1: 640px
# ======================================

# --- KIEM TRA LABELS TRUOC KHI TRAIN ---
# Loc bo labels co bbox qua nho hoac gia tri bat thuong
print('Sanitizing labels...')
sanitized = 0
removed_lines = 0
for split in ['train', 'val']:
    label_files = glob.glob(f'{WORK_DIR}/labels/{split}/*.txt')
    for lf in label_files:
        if os.path.getsize(lf) == 0:
            continue
        with open(lf, 'r') as f:
            lines = f.readlines()
        clean_lines = []
        for line in lines:
            parts = line.strip().split()
            if len(parts) != 5:
                removed_lines += 1
                continue
            cls_id, cx, cy, w, h = int(parts[0]), float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
            # Bo bbox qua nho (< 0.5% anh) hoac gia tri bat thuong
            if w < 0.005 or h < 0.005 or w > 1.0 or h > 1.0:
                removed_lines += 1
                continue
            if cx < 0 or cx > 1 or cy < 0 or cy > 1:
                removed_lines += 1
                continue
            if cls_id < 0 or cls_id >= 14:
                removed_lines += 1
                continue
            clean_lines.append(line)
        if len(clean_lines) != len(lines):
            sanitized += 1
            with open(lf, 'w') as f:
                f.writelines(clean_lines)
print(f'Sanitized: {sanitized} files modified, {removed_lines} bad lines removed')

# --- TRAIN PHASE 1 ---
model = YOLO('yolo11m.pt')

results_p1 = model.train(
    data=yaml_path,
    epochs=25,
    imgsz=640,
    batch=16,
    patience=10,
    mosaic=0.0,
    mixup=0.0,
    flipud=0.0,
    fliplr=0.5,
    degrees=5,
    scale=0.2,
    optimizer='AdamW',
    lr0=0.002,
    lrf=0.01,
    cos_lr=True,
    warmup_epochs=5,
    warmup_bias_lr=0.01,
    warmup_momentum=0.5,
    amp=True,
    name='chexnet_yolo_hybrid_phase1',
    device=DEVICE,
    exist_ok=True,
    verbose=True,
)
print('Phase 1 hoan tat!')

In [ ]:
# ======================================
# PHASE 2: 1024px
# ======================================
best_phase1 = 'runs/detect/chexnet_yolo_hybrid_phase1/weights/best.pt'
if not os.path.exists(best_phase1):
    candidates = glob.glob('runs/detect/chexnet_yolo_hybrid_phase1*/weights/best.pt')
    if not candidates:
        raise FileNotFoundError('Khong tim thay model Phase 1!')
    best_phase1 = candidates[0]

print(f'Loading Phase 1 best: {best_phase1}')
model_p2 = YOLO(best_phase1)

results_p2 = model_p2.train(
    data=yaml_path,
    epochs=55,
    imgsz=1024,
    batch=8,
    patience=15,
    mosaic=0.0,
    mixup=0.0,
    flipud=0.0,
    fliplr=0.5,
    degrees=10,
    scale=0.3,
    optimizer='AdamW',
    cos_lr=True,
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=3,
    name='chexnet_yolo_hybrid_phase2',
    device=DEVICE,
    exist_ok=True,
    verbose=True,
)
print('Phase 2 hoan tat!')

In [ ]:
# ======================================
# RESUME - Chay cell nay neu training bi gian doan
# ======================================
# Bo comment khi can resume:

# from ultralytics import YOLO
# model_resume = YOLO('runs/detect/chexnet_yolo_hybrid_phase2/weights/last.pt')
# model_resume.train(resume=True)

## 7. Đánh giá với TTA + WBF

Dự đoán ảnh gốc + flip ngang, gộp bằng WBF.

In [ ]:
def tta_wbf_predict(model, img_path, imgsz=1024, conf=0.01, iou_thr=0.4):
    """TTA inference: anh goc + flip ngang, gop bang WBF."""
    img = cv2.imread(img_path)
    img_flip = cv2.flip(img, 1)
    
    res_orig = model.predict(img, imgsz=imgsz, conf=conf, verbose=False)[0]
    res_flip = model.predict(img_flip, imgsz=imgsz, conf=conf, verbose=False)[0]
    
    b1 = res_orig.boxes.xyxyn.cpu().numpy().tolist() if len(res_orig.boxes) > 0 else []
    s1 = res_orig.boxes.conf.cpu().numpy().tolist() if len(res_orig.boxes) > 0 else []
    l1 = res_orig.boxes.cls.cpu().numpy().astype(int).tolist() if len(res_orig.boxes) > 0 else []
    
    if len(res_flip.boxes) > 0:
        b2_raw = res_flip.boxes.xyxyn.cpu().numpy().copy()
        b2_raw[:, [0, 2]] = 1.0 - b2_raw[:, [2, 0]]
        b2, s2, l2 = b2_raw.tolist(), res_flip.boxes.conf.cpu().numpy().tolist(), res_flip.boxes.cls.cpu().numpy().astype(int).tolist()
    else:
        b2, s2, l2 = [], [], []
    
    if not b1 and not b2:
        return np.array([]), np.array([]), np.array([])
    
    all_boxes = [b1 if b1 else [[0,0,0,0]], b2 if b2 else [[0,0,0,0]]]
    all_scores = [s1 if s1 else [0], s2 if s2 else [0]]
    all_labels = [l1 if l1 else [0], l2 if l2 else [0]]
    
    fused_boxes, fused_scores, fused_labels = weighted_boxes_fusion(
        all_boxes, all_scores, all_labels, weights=[1, 1],
        iou_thr=iou_thr, skip_box_thr=0.01
    )
    return fused_boxes, fused_scores, fused_labels

# -- Danh gia --
best_p2 = 'runs/detect/chexnet_yolo_hybrid_phase2/weights/best.pt'
if os.path.exists(best_p2):
    best_model = YOLO(best_p2)
    print('Standard evaluation:')
    metrics = best_model.val(data=yaml_path, imgsz=1024)
    print(f'mAP50: {metrics.box.map50:.4f}')
    print(f'mAP50-95: {metrics.box.map:.4f}')
    print('\nPer-class AP50:')
    for i, name in enumerate(YOLO_CLASSES):
        if i < len(metrics.box.ap50):
            print(f'  {name:25s}: {metrics.box.ap50[i]:.4f}')
else:
    print('Chua co model Phase 2.')

## 8. Cascade 2 giai đoạn + Mapping

`Final_Score = P_CheXNet_Classifier * Conf_YOLO_Detector`

YOLO class ID da dong bo 1-1 voi CheXNet (YOLO_id + 1 = CheXNet_id).

In [ ]:
def cascade_filter(yolo_boxes, yolo_confs, yolo_classes, chexnet_probs):
    """Loc YOLO boxes bang xac suat tu CheXNet classifier."""
    CLINICAL_THRESHOLDS = {
        6:  0.05,  # Pneumonia
        7:  0.05,  # Pneumothorax
        1:  0.10,  # Cardiomegaly
        2:  0.10,  # Effusion
        8:  0.12,  # Consolidation
    }
    DEFAULT_THRESHOLD = 0.15
    
    filtered = {'boxes': [], 'confs': [], 'classes': []}
    for box, conf, cls_id in zip(yolo_boxes, yolo_confs, yolo_classes):
        cls_id = int(cls_id)
        p_class = chexnet_probs.get(cls_id, 0.5)
        final_score = float(conf) * p_class
        thresh = CLINICAL_THRESHOLDS.get(cls_id, DEFAULT_THRESHOLD)
        if final_score >= thresh:
            filtered['boxes'].append(box)
            filtered['confs'].append(final_score)
            filtered['classes'].append(cls_id)
    return filtered

# -- Mapping YOLO <-> CheXNet --
# YOLO cls_id + 1 = CheXNet class index (vi CheXNet co No Finding o idx 0)
YOLO_TO_CHEXNET_IDX = {i: i + 1 for i in range(14)}
print('YOLO <-> CheXNet mapping:')
for yolo_id, chex_id in YOLO_TO_CHEXNET_IDX.items():
    print(f'  YOLO {yolo_id:2d} ({YOLO_CLASSES[yolo_id]:25s}) <-> CheXNet {chex_id:2d} ({CHEXNET_CLASS_NAMES[chex_id]})')

In [ ]:
# ======================================
# Export model tot nhat
# ======================================
best_paths = [
    'runs/detect/chexnet_yolo_hybrid_phase2/weights/best.pt',
    'runs/detect/chexnet_yolo_hybrid_phase1/weights/best.pt',
]

exported = False
for bp in best_paths:
    if os.path.exists(bp):
        out_path = '/kaggle/working/yolo11m_hybrid.pt'
        shutil.copy(bp, out_path)
        print(f'Model exported: {bp} -> {out_path}')
        m = YOLO(out_path)
        m.info()
        exported = True
        break

if not exported:
    print('Chua co model trained.')

## 9. Tổng kết

### Đã thực hiện:
- Knowledge Distillation: CheXNet Attention Map sinh Pseudo BBox (Pneumonia, Edema, Emphysema, Hernia)
- Dataset Hybrid: VinDr-CXR GT + NIH Pseudo-labels, 14 classes đồng bộ CheXNet
- WBF trên Ground Truth (gộp nhiều bác sĩ thành consensus box)
- Image Path Map O(1) quét 10 thư mục data_*
- Symlink tiết kiệm disk (không copy 50GB ảnh)
- Progressive Training (640px rồi 1024px)
- TTA + WBF Inference
- Cascade 2 giai đoạn (CheXNet x YOLO)
- YOLO - CheXNet mapping đồng bộ (class name 1-1)

### Bước tiếp theo:
1. Tải file `yolo11m_hybrid.pt` về máy
2. Đặt vào `CheXNet/Trainedmodel/yolov11m.pt`
3. Cập nhật `Backend/yolo_detector.py` với mapping 14 class mới
4. Cập nhật `Backend/main.py` để render heatmap BÊN TRONG YOLO bbox